# Legacy vs H121 — OFA comparison over June 2020 (innov_newQC)

Compares Legacy BUFR ASCAT (species 9/10/11) against H121 CDR ASCAT (species 14/15/16) as written
to ObsFcstAna, using the full June 2020 `hsaf_cdr_test_DAv8_M36_202006_innov_newQC` run — the run
we're treating as the new default (see `innov_vs_innov_newqc.ipynb` for why).

July 2020 OFA data isn't downloaded yet, so this notebook uses the full June window instead
(2020-06-01 through 2020-06-30; June 1 and June 30 are partial days at the run's edges).

Both products come from the *same* run/OFA files here — this is a legacy-vs-H121 product
comparison, not a QC-variant comparison.


In [ ]:
import sys, os
from pathlib import Path


def _find_lib_root():
    cwd = Path(os.path.abspath(''))
    for p in [cwd] + list(cwd.parents):
        if (p / 'lib').exists() and (p / 'lib' / 'readers.py').exists():
            return p
        for child in p.glob('projects/*/lib'):
            if (child / 'readers.py').exists():
                return child.parent
    raise RuntimeError(f'Cannot find ascat_da/lib/ from {cwd}')


_root = _find_lib_root()
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

_repo_root = Path(_root).parents[1]
_common_io = _repo_root / 'common' / 'python' / 'io'
if str(_common_io) not in sys.path:
    sys.path.insert(0, str(_common_io))
from read_GEOSldas import read_tilecoord

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature


In [ ]:
# ── Configuration — edit here ─────────────────────────────────────────────────
START_DATE = '2020-06-01'
END_DATE   = '2020-06-30'

CACHE_VERSION = 'newqc'
CACHE_DIR = Path(_root) / '.cache' / 'ofa'
CACHE_TAG = f"{START_DATE.replace('-', '')}_{END_DATE.replace('-', '')}_{CACHE_VERSION}"

PRODUCTS = {
    'legacy': {'Metop-A': 9,  'Metop-B': 10, 'Metop-C': 11},
    'h121':   {'Metop-A': 14, 'Metop-B': 15, 'Metop-C': 16},
}
PLATFORM_COLOR = {'Metop-A': '#1f77b4', 'Metop-B': '#ff7f0e', 'Metop-C': '#2ca02c'}

# Tile lat/lon must come from the tilecoord file, not the OFA file's own lat/lon
# (the latter is the per-cycle super-ob center and jitters slightly within a tile).
TILE_BASE = '/Users/amfox/Desktop/ASCAT_SSM_CDR/discover_sample/tilecoord/hsaf_cdr_test_DAv8_M36_202006_innov'
TILECOORD = f'{TILE_BASE}.ldas_tilecoord.bin'

print(f"Date range: {START_DATE} .. {END_DATE}")
print(f"Cache tag: {CACHE_TAG}")


## 1. Load cached OFA tile/cycle table

Pre-aggregated with `build_ofa_cache.py` (one row per date/cycle/species/tile):

```bash
python projects/ascat_da/scripts/build_ofa_cache.py \
  --start-date 2020-06-01 --end-date 2020-06-30 \
  --ofa-dir data/hsaf_cdr_test/hsaf_cdr_test_DAv8_M36_202006_innov_newQC/output/SMAP_EASEv2_M36_GLOBAL/ana/ens_avg/Y2020/M06 \
  --out-dir projects/ascat_da/.cache/ofa \
  --version newqc
```


In [ ]:
pkl = CACHE_DIR / f'ofa_ascat_tile_cycle_{CACHE_TAG}.pkl'
if not pkl.exists():
    raise FileNotFoundError(f'Missing OFA cache: {pkl}\nRun build_ofa_cache.py (see markdown above).')

ofa = pd.read_pickle(pkl)
print(f"{len(ofa):,} tile/cycle rows, {ofa['date'].min()} .. {ofa['date'].max()}")

tile_coord = read_tilecoord(TILECOORD)
tile_latlon = pd.DataFrame({
    'tilenum': tile_coord['tile_id'].astype('int64'),
    'tile_lat': tile_coord['com_lat'],
    'tile_lon': tile_coord['com_lon'],
}).drop_duplicates('tilenum')

ofa = ofa.merge(tile_latlon, on='tilenum', how='left')
n_missing = ofa['tile_lat'].isna().sum()
if n_missing:
    print(f"WARNING: {n_missing} OFA rows have no matching tilecoord entry")


## 2. Observation counts: legacy vs H121

Raw super-ob counts by product/platform over the full month.


In [ ]:
counts = (
    ofa.groupby(['product', 'platform'], as_index=False)
    .agg(total_ofa_obs=('tilenum', 'size'))
)
pivot = counts.pivot_table(index='platform', columns='product', values='total_ofa_obs')
pivot['h121_to_legacy_ratio'] = pivot['h121'] / pivot['legacy']
display(pivot.round(2))

fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(len(pivot))
width = 0.35
ax.bar(x - width / 2, pivot['legacy'], width, label='legacy', color='#555555')
ax.bar(x + width / 2, pivot['h121'], width, label='h121', color='#2ca02c')
ax.set_xticks(x)
ax.set_xticklabels(pivot.index)
ax.set_ylabel('Total OFA obs (June 2020)')
ax.set_title('Obs counts: legacy vs H121')
ax.legend()
fig.tight_layout()


## 3. Maps: per-tile obs counts (legacy vs H121)

Per-tile obs counts summed over June, by platform (columns). Rows: legacy, H121, and H121-minus-legacy.


In [ ]:
platforms = ['Metop-A', 'Metop-B', 'Metop-C']


def tile_counts(df, species_id):
    sub = df[df['species'] == species_id]
    return (
        sub.groupby('tilenum', as_index=False)
        .agg(lat=('tile_lat', 'first'), lon=('tile_lon', 'first'), n_obs=('tilenum', 'size'))
    )


tile_maps = {}
for plat in platforms:
    legacy_t = tile_counts(ofa, PRODUCTS['legacy'][plat])
    h121_t = tile_counts(ofa, PRODUCTS['h121'][plat])
    merged = legacy_t.merge(h121_t, on='tilenum', how='outer', suffixes=('_legacy', '_h121'))
    merged['lat'] = merged['lat_legacy'].combine_first(merged['lat_h121'])
    merged['lon'] = merged['lon_legacy'].combine_first(merged['lon_h121'])
    merged['n_obs_legacy'] = merged['n_obs_legacy'].fillna(0)
    merged['n_obs_h121'] = merged['n_obs_h121'].fillna(0)
    merged['diff'] = merged['n_obs_h121'] - merged['n_obs_legacy']
    tile_maps[plat] = merged

from matplotlib.colors import LogNorm, TwoSlopeNorm

count_norm = LogNorm(vmin=1, vmax=max(m[['n_obs_legacy', 'n_obs_h121']].values.max() for m in tile_maps.values()))
diff_abs_max = max(m['diff'].abs().max() for m in tile_maps.values())
diff_norm = TwoSlopeNorm(vmin=-diff_abs_max, vcenter=0, vmax=diff_abs_max)

fig, axes = plt.subplots(3, 3, figsize=(15, 10), subplot_kw={'projection': ccrs.Robinson()})
row_specs = [
    ('n_obs_legacy', 'legacy', 'viridis', count_norm),
    ('n_obs_h121', 'h121', 'viridis', count_norm),
    ('diff', 'h121 - legacy', 'RdBu_r', diff_norm),
]

for row, (col, row_label, cmap, norm) in enumerate(row_specs):
    for ax_col, plat in enumerate(platforms):
        ax = axes[row, ax_col]
        m = tile_maps[plat]
        sc = ax.scatter(
            m['lon'], m['lat'], c=m[col], s=0.5, cmap=cmap, norm=norm,
            transform=ccrs.PlateCarree(),
        )
        ax.add_feature(cfeature.COASTLINE, linewidth=0.4)
        ax.set_extent([-180, 180, -60, 85], crs=ccrs.PlateCarree())
        if row == 0:
            ax.set_title(plat, fontsize=11)
        if ax_col == 0:
            ax.text(-0.08, 0.5, row_label, transform=ax.transAxes, rotation=90,
                     va='center', ha='center', fontsize=10)
    fig.colorbar(sc, ax=axes[row, :].tolist(), shrink=0.7, pad=0.01,
                 label='obs count' if row < 2 else 'Δ obs count')

fig.suptitle('Legacy vs H121 obs count per tile (June 2020 total)', y=0.99)


In [ ]:
def tile_value_map(df, species_id, value_col, agg='mean'):
    sub = df[df['species'] == species_id].copy()
    if agg == 'mean_abs':
        sub['_val'] = sub[value_col].abs()
        agg_func = 'mean'
    else:
        sub['_val'] = sub[value_col]
        agg_func = agg
    return (
        sub.groupby('tilenum', as_index=False)
        .agg(lat=('tile_lat', 'first'), lon=('tile_lon', 'first'), value=('_val', agg_func))
    )


def plot_legacy_h121_diff_maps(value_col, agg, title, cbar_label, diff_label='h121 - legacy'):
    tmaps = {}
    for plat in platforms:
        legacy_t = tile_value_map(ofa, PRODUCTS['legacy'][plat], value_col, agg)
        h121_t = tile_value_map(ofa, PRODUCTS['h121'][plat], value_col, agg)
        merged = legacy_t.merge(h121_t, on='tilenum', how='outer', suffixes=('_legacy', '_h121'))
        merged['lat'] = merged['lat_legacy'].combine_first(merged['lat_h121'])
        merged['lon'] = merged['lon_legacy'].combine_first(merged['lon_h121'])
        merged['diff'] = merged['value_h121'] - merged['value_legacy']
        tmaps[plat] = merged

    vmin = min(m[['value_legacy', 'value_h121']].min().min() for m in tmaps.values())
    vmax = max(m[['value_legacy', 'value_h121']].max().max() for m in tmaps.values())
    diff_abs_max = max(m['diff'].abs().max() for m in tmaps.values())
    diff_norm = TwoSlopeNorm(vmin=-diff_abs_max, vcenter=0, vmax=diff_abs_max)

    fig, axes = plt.subplots(3, 3, figsize=(15, 10), subplot_kw={'projection': ccrs.Robinson()})
    row_specs = [
        ('value_legacy', 'legacy', 'viridis', dict(vmin=vmin, vmax=vmax)),
        ('value_h121', 'h121', 'viridis', dict(vmin=vmin, vmax=vmax)),
        ('diff', diff_label, 'RdBu_r', dict(norm=diff_norm)),
    ]
    for row, (col, row_label, cmap, norm_kw) in enumerate(row_specs):
        for ax_col, plat in enumerate(platforms):
            ax = axes[row, ax_col]
            m = tmaps[plat]
            sc = ax.scatter(
                m['lon'], m['lat'], c=m[col], s=0.5, cmap=cmap,
                transform=ccrs.PlateCarree(), **norm_kw,
            )
            ax.add_feature(cfeature.COASTLINE, linewidth=0.4)
            ax.set_extent([-180, 180, -60, 85], crs=ccrs.PlateCarree())
            if row == 0:
                ax.set_title(plat, fontsize=11)
            if ax_col == 0:
                ax.text(-0.08, 0.5, row_label, transform=ax.transAxes, rotation=90,
                         va='center', ha='center', fontsize=10)
        fig.colorbar(sc, ax=axes[row, :].tolist(), shrink=0.7, pad=0.01,
                     label=cbar_label if row < 2 else f'Δ {cbar_label}')
    fig.suptitle(title, y=0.99)
    return tmaps


## 4. Maps: mean obs value (legacy vs H121)

Per-tile mean obs over June (degree-of-saturation %, model-space units), rows = legacy /
H121 / diff, columns = Metop-A/B/C.


In [ ]:
_ = plot_legacy_h121_diff_maps(
    'obs_pct', 'mean', 'Mean obs per tile (June 2020): legacy vs H121', 'mean obs',
)


## 5. Maps: mean |innovation| (legacy vs H121)

Per-tile mean absolute innov (mean(|obs - fcst|)) over June — a measure of typical
innovation magnitude, independent of sign cancellation.


In [ ]:
_ = plot_legacy_h121_diff_maps(
    'innov_pct', 'mean_abs', 'Mean |innov| per tile (June 2020): legacy vs H121',
    'mean |innov|', diff_label='h121 - legacy (|innov|)',
)


## 6. Coverage fraction (resolution-agnostic)

As in `innov_vs_innov_newqc.ipynb`: raw counts aren't directly comparable across products with
different swath/footprint resolutions. Coverage fraction = share of June's analysis cycles
(30 days x 8 cycles = 240) where *any* platform of that product produced a super-ob for the tile.


In [ ]:
N_CYCLES_TOTAL = len(pd.date_range(START_DATE, END_DATE)) * 8


def coverage_fraction(df, species_list, n_total):
    sub = (
        df[df['species'].isin(species_list)][['tilenum', 'date', 'cycle', 'tile_lat', 'tile_lon']]
        .drop_duplicates(['tilenum', 'date', 'cycle'])
    )
    cov = sub.groupby('tilenum').agg(
        n_cycles=('date', 'size'), lat=('tile_lat', 'first'), lon=('tile_lon', 'first'),
    )
    cov['coverage_frac'] = cov['n_cycles'] / n_total
    return cov.reset_index()


legacy_species = list(PRODUCTS['legacy'].values())
h121_species = list(PRODUCTS['h121'].values())

cov_legacy = coverage_fraction(ofa, legacy_species, N_CYCLES_TOTAL)
cov_h121 = coverage_fraction(ofa, h121_species, N_CYCLES_TOTAL)

cov_panels = [
    ('Legacy (Metop-A/B/C)', cov_legacy),
    ('H121 (Metop-A/B/C)', cov_h121),
]

fig, axes = plt.subplots(2, 1, figsize=(22, 10), subplot_kw={'projection': ccrs.Robinson()})
for ax, (title, cov) in zip(axes, cov_panels):
    sc = ax.scatter(
        cov['lon'], cov['lat'], c=cov['coverage_frac'], s=0.5, cmap='viridis',
        vmin=0, vmax=1, transform=ccrs.PlateCarree(),
    )
    ax.add_feature(cfeature.COASTLINE, linewidth=0.4)
    ax.set_extent([-180, 180, -60, 85], crs=ccrs.PlateCarree())
    ax.set_title(f"{title}  (mean={cov['coverage_frac'].mean():.2f})", fontsize=11)
fig.colorbar(sc, ax=axes.tolist(), shrink=0.7, pad=0.02, label='fraction of cycles with a super-ob')
fig.suptitle('Per-tile coverage fraction — June 2020', y=1.01)


## 7. Distribution comparison: obs and innov

Legacy and H121 obs are both in degree-of-saturation % (model-space units after scaling), so
direct distributional comparison is meaningful here — unlike the raw counts above.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 7), sharex='row')
for col, plat in enumerate(platforms):
    legacy_obs = ofa.loc[ofa['species'] == PRODUCTS['legacy'][plat], 'obs_pct'].dropna()
    h121_obs = ofa.loc[ofa['species'] == PRODUCTS['h121'][plat], 'obs_pct'].dropna()
    axes[0, col].hist(legacy_obs, bins=60, alpha=0.6, label='legacy', color='#555555', density=True)
    axes[0, col].hist(h121_obs, bins=60, alpha=0.6, label='h121', color='#2ca02c', density=True)
    axes[0, col].set_title(f'{plat}: obs')

    legacy_innov = ofa.loc[ofa['species'] == PRODUCTS['legacy'][plat], 'innov_pct'].dropna()
    h121_innov = ofa.loc[ofa['species'] == PRODUCTS['h121'][plat], 'innov_pct'].dropna()
    axes[1, col].hist(legacy_innov, bins=60, alpha=0.6, label='legacy', color='#555555', density=True)
    axes[1, col].hist(h121_innov, bins=60, alpha=0.6, label='h121', color='#2ca02c', density=True)
    axes[1, col].set_title(f'{plat}: innov')

axes[0, 0].legend()
fig.tight_layout()


## 8. Matched scatter: legacy vs H121 obs values

Match legacy and H121 super-obs on (date, cycle, tilenum) per platform — same satellite, same
analysis window, same model tile — so the comparison is of the actual observation value, not
just aggregate distributions. Only tile-cycles where *both* products reported a super-ob are kept.


In [ ]:
def _scatter_stats(x, y):
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    keep = np.isfinite(x) & np.isfinite(y)
    x, y = x[keep], y[keep]
    if len(x) == 0:
        return dict(n=0, bias=np.nan, rmse=np.nan, r=np.nan)
    diff = y - x
    return dict(
        n=len(x),
        bias=float(np.mean(diff)),
        rmse=float(np.sqrt(np.mean(diff ** 2))),
        r=float(np.corrcoef(x, y)[0, 1]) if len(x) > 1 else np.nan,
    )


def _sample_for_plot(df, max_points=80000, random_state=42):
    if len(df) <= max_points:
        return df
    return df.sample(max_points, random_state=random_state)


MATCH_KEYS = ['date', 'cycle', 'tilenum']

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
matched_obs = {}
for ax, plat in zip(axes, platforms):
    legacy_sub = ofa.loc[ofa['species'] == PRODUCTS['legacy'][plat], MATCH_KEYS + ['obs_pct']]
    h121_sub = ofa.loc[ofa['species'] == PRODUCTS['h121'][plat], MATCH_KEYS + ['obs_pct']]
    matched = legacy_sub.merge(h121_sub, on=MATCH_KEYS, suffixes=('_legacy', '_h121'))
    matched = matched.merge(tile_latlon[['tilenum', 'tile_lat']], on='tilenum', how='left')
    matched_obs[plat] = matched

    stats = _scatter_stats(matched['obs_pct_legacy'], matched['obs_pct_h121'])
    plot_df = _sample_for_plot(matched)
    sc = ax.scatter(
        plot_df['obs_pct_legacy'], plot_df['obs_pct_h121'], s=2, alpha=0.3,
        c=plot_df['tile_lat'], cmap='coolwarm', vmin=-60, vmax=85,
    )
    ax.plot([0, 100], [0, 100], 'k--', linewidth=1)
    ax.set_xlim(0, 100)
    ax.set_ylim(0, 100)
    ax.set_xlabel('legacy obs')
    ax.set_ylabel('h121 obs')
    ax.set_title(
        f"{plat}  (n={stats['n']:,})\n"
        f"bias={stats['bias']:.2f}  rmse={stats['rmse']:.2f}  r={stats['r']:.2f}"
    )

fig.tight_layout(rect=[0, 0, 0.93, 1])
cax = fig.add_axes([0.95, 0.15, 0.012, 0.7])
fig.colorbar(sc, cax=cax, label='tile latitude')
fig.suptitle('Matched legacy vs H121 obs values (same date/cycle/tile) — June 2020', y=1.03)


## 9. Maps: goodness of fit (matched legacy vs H121)

Per-tile fit statistics computed from the matched pairs in Section 8: correlation (r), bias
(h121 - legacy), and RMSE. Tiles with fewer than `MIN_MATCHED_N` matched cycles are masked out —
per-tile r is unreliable on small samples — and shown separately as a sample-count map so the
gaps are visible rather than silently dropped.


In [ ]:
MIN_MATCHED_N = 10


def tile_fit_stats(matched, min_n=MIN_MATCHED_N):
    def _stats(g):
        x = g['obs_pct_legacy'].to_numpy(float)
        y = g['obs_pct_h121'].to_numpy(float)
        n = len(g)
        diff = y - x
        r = np.corrcoef(x, y)[0, 1] if n >= 2 and np.std(x) > 0 and np.std(y) > 0 else np.nan
        return pd.Series({
            'n': n,
            'r': r,
            'bias': diff.mean(),
            'rmse': np.sqrt((diff ** 2).mean()),
        })

    stats = matched.groupby('tilenum').apply(_stats, include_groups=False).reset_index()
    stats = stats.merge(tile_latlon, on='tilenum', how='left')
    stats.loc[stats['n'] < min_n, ['r', 'bias', 'rmse']] = np.nan
    return stats


fit_stats = {plat: tile_fit_stats(matched_obs[plat]) for plat in platforms}

fig, axes = plt.subplots(3, 3, figsize=(15, 10), subplot_kw={'projection': ccrs.Robinson()})
row_specs = [
    ('r', 'correlation (r)', 'viridis', dict(vmin=-1, vmax=1)),
    ('bias', 'bias (h121-legacy)', 'RdBu_r', dict(vmin=-20, vmax=20)),
    ('rmse', 'RMSE', 'magma', dict(vmin=0, vmax=30)),
]
for row, (col, row_label, cmap, norm_kw) in enumerate(row_specs):
    for ax_col, plat in enumerate(platforms):
        ax = axes[row, ax_col]
        s = fit_stats[plat]
        sc = ax.scatter(
            s['tile_lon'], s['tile_lat'], c=s[col], s=0.5, cmap=cmap,
            transform=ccrs.PlateCarree(), **norm_kw,
        )
        ax.add_feature(cfeature.COASTLINE, linewidth=0.4)
        ax.set_extent([-180, 180, -60, 85], crs=ccrs.PlateCarree())
        if row == 0:
            ax.set_title(plat, fontsize=11)
        if ax_col == 0:
            ax.text(-0.08, 0.5, row_label, transform=ax.transAxes, rotation=90,
                     va='center', ha='center', fontsize=10)
    fig.colorbar(sc, ax=axes[row, :].tolist(), shrink=0.7, pad=0.01, label=row_label)
fig.suptitle(f'Matched legacy vs H121 goodness of fit per tile (min n={MIN_MATCHED_N})', y=0.99)


Sample-count map (matched cycles per tile) — for context on where the fit stats above are
well- vs poorly-sampled.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), subplot_kw={'projection': ccrs.Robinson()})
n_max = max(s['n'].max() for s in fit_stats.values())
for ax, plat in zip(axes, platforms):
    s = fit_stats[plat]
    sc = ax.scatter(
        s['tile_lon'], s['tile_lat'], c=s['n'], s=0.5, cmap='viridis',
        norm=LogNorm(vmin=1, vmax=n_max), transform=ccrs.PlateCarree(),
    )
    ax.add_feature(cfeature.COASTLINE, linewidth=0.4)
    ax.set_extent([-180, 180, -60, 85], crs=ccrs.PlateCarree())
    ax.set_title(plat, fontsize=11)
fig.colorbar(sc, ax=axes.tolist(), shrink=0.7, pad=0.02, label='matched cycles per tile')
fig.suptitle('Matched-pair sample count per tile', y=1.03)


## 10. Daily obs counts over June

Daily total obs by product — checks for swath/orbit-related dips or trends within the month,
separate from the QC-driven shifts already characterized in `innov_vs_innov_newqc.ipynb`.


In [ ]:
daily = (
    ofa.groupby(['date', 'product'], as_index=False)
    .agg(n_obs=('tilenum', 'size'))
    .pivot(index='date', columns='product', values='n_obs')
)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(daily.index, daily['legacy'], label='legacy', color='#555555', marker='o', markersize=3)
ax.plot(daily.index, daily['h121'], label='h121', color='#2ca02c', marker='o', markersize=3)
ax.set_ylabel('Daily obs count (all platforms)')
ax.set_title('Daily obs counts: legacy vs H121 — June 2020')
ax.tick_params(axis='x', rotation=45)
ax.legend()
fig.tight_layout()


## 11. Picking 2-deg target boxes in other regions

Want the same regional window-map view for Australia, South America, Africa, Western Europe,
and boreal Russia, without manually hunting for a good box in each. Reuse `fit_stats['Metop-A']`
from section 9 (already computed: per-tile matched Legacy/H121 cycle counts over all of June) --
bin it onto a 2-deg grid within each region's rough bounding box, and pick the 2-deg cell with
the most matched tile/cycles. That guarantees the box actually has data worth plotting without
a separate search.

In [ ]:
MACRO_REGIONS = {
    'Australia':        (-44, 112, -10, 154),
    'South America':    (-55, -82,  12, -34),
    'Africa':           (-35, -18,  38,  52),
    'Western Europe':   (36,  -10,  60,  20),
    'Boreal Russia':    (55,   30,  70, 160),
}
BOX_SIZE = 2.0

def best_box(stats, lat0, lon0, lat1, lon1, box_size=BOX_SIZE):
    s = stats[stats['tile_lat'].between(lat0, lat1) & stats['tile_lon'].between(lon0, lon1)].copy()
    if len(s) == 0 or s['n'].sum() == 0:
        return None
    s['lat_box'] = (np.floor(s['tile_lat'] / box_size) * box_size)
    s['lon_box'] = (np.floor(s['tile_lon'] / box_size) * box_size)
    by_box = s.groupby(['lat_box', 'lon_box'], as_index=False)['n'].sum().sort_values('n', ascending=False)
    top = by_box.iloc[0]
    return (float(top['lat_box']), float(top['lon_box']), float(top['lat_box'] + box_size), float(top['lon_box'] + box_size), int(top['n']))

auto_boxes = {}
for name, (lat0, lon0, lat1, lon1) in MACRO_REGIONS.items():
    result = best_box(fit_stats['Metop-A'], lat0, lon0, lat1, lon1)
    if result is None:
        print(f'{name}: no matched tiles found in macro-region -- skipping')
        continue
    auto_boxes[name] = result[:4]
    print(f"{name:16s}  box=({result[0]:.0f}, {result[1]:.0f}, {result[2]:.0f}, {result[3]:.0f})  matched tile-cycles in box={result[4]:,}")

ALL_REGIONS = {'Oklahoma / S Kansas': (35, -99, 38, -96), **auto_boxes}


## 12. Regional window maps across target regions

A global map first, showing where each box sits, then the same window-selection and tile-plotting machinery as the Oklahoma example, applied to Oklahoma plus the 5 auto-selected boxes. Each window's title includes the matched-pair goodness-of-fit (r, bias, RMSE) for that specific date/cycle/region.

In [ ]:
fig = plt.figure(figsize=(13, 6))
ax = plt.axes(projection=ccrs.Robinson())
ax.add_feature(cfeature.LAND, facecolor="0.85", zorder=0)
ax.add_feature(cfeature.OCEAN, facecolor="white", zorder=0)
ax.add_feature(cfeature.COASTLINE, linewidth=0.4, zorder=1)
ax.set_global()

for name, (lat0, lon0, lat1, lon1) in ALL_REGIONS.items():
    clat, clon = (lat0 + lat1) / 2, (lon0 + lon1) / 2
    ax.plot(clon, clat, marker="*", markersize=14, color="#d62728", markeredgecolor="black",
            markeredgewidth=0.5, transform=ccrs.PlateCarree(), zorder=3)
    ax.text(clon, clat - 6, name, transform=ccrs.PlateCarree(),
            ha="center", va="top", fontsize=8.5, fontweight="bold", zorder=4,
            bbox=dict(facecolor="white", edgecolor="none", alpha=0.7, pad=1.5))
ax.set_title("Regional window-map target boxes (2-deg each, marker at box center)")
fig.tight_layout()


In [ ]:
import matplotlib as mpl
import matplotlib.patches as mpatches
from matplotlib.collections import PatchCollection

from lib.superob import ij_to_corners

REGION_PLATFORM = 'Metop-A'
N_REGION_WINDOWS = 10

tile_grid_df = pd.DataFrame({
    'tilenum': tile_coord['tile_id'].astype('int64'),
    'tile_lat': tile_coord['com_lat'],
    'tile_lon': tile_coord['com_lon'],
    'tile_i': tile_coord['i_indg'].astype('int64'),
    'tile_j': tile_coord['j_indg'].astype('int64'),
}).drop_duplicates('tilenum')


def _setup_region_ax(ax, lat0, lon0, lat1, lon1, title=None):
    pad = 0.35
    ax.set_extent([lon0 - pad, lon1 + pad, lat0 - pad, lat1 + pad], crs=ccrs.PlateCarree())
    ax.set_facecolor('#f7f7f7')
    ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
    ax.add_feature(cfeature.BORDERS, linewidth=0.5, edgecolor='0.35')
    ax.add_feature(cfeature.STATES, linewidth=0.4, edgecolor='0.5')
    gl = ax.gridlines(draw_labels=True, linewidth=0.25, color='0.70', alpha=0.45)
    gl.top_labels = False
    gl.right_labels = False
    gl.xlabel_style = {'size': 7}
    gl.ylabel_style = {'size': 7}
    if title:
        ax.set_title(title)


def _draw_tile_values(ax, data, value_col, cmap, norm):
    data = data[np.isfinite(data[value_col])]
    if len(data) == 0:
        return None
    patches, values = [], []
    for _, r in data.iterrows():
        clons, clats = ij_to_corners(int(r['tile_i']), int(r['tile_j']))
        patches.append(mpatches.Polygon(np.column_stack([clons, clats]), closed=True))
        values.append(float(r[value_col]))
    coll = PatchCollection(
        patches, cmap=cmap, norm=norm, edgecolor='#555', linewidth=0.3,
        transform=ccrs.PlateCarree(), zorder=4,
    )
    coll.set_array(np.asarray(values, dtype=float))
    ax.add_collection(coll)
    return coll


def pick_windows(ofa_region, n_windows=N_REGION_WINDOWS):
    """One window every ~3 days: prefer the target date if it has H121 obs in-region,
    else the next day, else the previous day, else skip."""
    window_counts = (
        ofa_region.groupby(['date', 'cycle', 'product'], as_index=False)
        .agg(n_obs=('tilenum', 'size'))
    )
    total_by_window = window_counts.groupby(['date', 'cycle'], as_index=False)['n_obs'].sum()
    h121_dates = set(window_counts.loc[window_counts['product'] == 'h121', 'date'])

    target_dates = pd.date_range(START_DATE, END_DATE, periods=n_windows).strftime('%Y-%m-%d')
    chosen = []
    for target in target_dates:
        target_ts = pd.Timestamp(target)
        candidates = [target, (target_ts + pd.Timedelta(days=1)).strftime('%Y-%m-%d'),
                      (target_ts - pd.Timedelta(days=1)).strftime('%Y-%m-%d')]
        picked_date = next((d for d in candidates if d in h121_dates), None)
        if picked_date is None:
            continue
        day_windows = total_by_window[total_by_window['date'] == picked_date]
        best = day_windows.loc[day_windows['n_obs'].idxmax()]
        chosen.append({'date': picked_date, 'cycle': int(best['cycle']), 'n_obs': int(best['n_obs'])})
    return pd.DataFrame(chosen)


def plot_regional_windows(region_name, lat0, lon0, lat1, lon1, platform=REGION_PLATFORM):
    region_tiles = tile_grid_df[tile_grid_df['tile_lat'].between(lat0, lat1) & tile_grid_df['tile_lon'].between(lon0, lon1)]
    ofa_region = ofa[
        (ofa['platform'] == platform) & ofa['tilenum'].isin(region_tiles['tilenum'])
    ].merge(region_tiles[['tilenum', 'tile_i', 'tile_j']], on='tilenum', how='left')

    print(f"{region_name}: {len(region_tiles):,} tiles, {len(ofa_region):,} {platform} OFA rows in box ({lat0:.0f},{lon0:.0f},{lat1:.0f},{lon1:.0f})")
    windows = pick_windows(ofa_region)
    if windows.empty:
        print(f'{region_name}: no usable windows -- skipping')
        return

    value_norm = mpl.colors.Normalize(0, 100)
    for _, win in windows.iterrows():
        date, cycle = win['date'], int(win['cycle'])
        subset = ofa_region[(ofa_region['date'] == date) & (ofa_region['cycle'] == cycle)]

        legacy_p = subset.loc[subset['product'] == 'legacy', ['tilenum', 'obs_pct']]
        h121_p = subset.loc[subset['product'] == 'h121', ['tilenum', 'obs_pct']]
        matched = legacy_p.merge(h121_p, on='tilenum', suffixes=('_legacy', '_h121'))
        st = _scatter_stats(matched['obs_pct_legacy'], matched['obs_pct_h121'])
        stats_str = f"matched n={st['n']}, r={st['r']:.2f}, bias={st['bias']:+.1f}, RMSE={st['rmse']:.1f}"

        fig, axes = plt.subplots(1, 2, figsize=(10, 4.6), subplot_kw={'projection': ccrs.PlateCarree()}, constrained_layout=True)
        for ax, product, title in zip(axes, ['legacy', 'h121'], ['Legacy', 'H121']):
            p = subset[subset['product'] == product]
            _setup_region_ax(ax, lat0, lon0, lat1, lon1, f'{title} (n={len(p)})')
            _draw_tile_values(ax, p, 'obs_pct', plt.cm.viridis, value_norm)
        fig.colorbar(mpl.cm.ScalarMappable(norm=value_norm, cmap='viridis'), ax=axes, orientation='horizontal',
                     fraction=0.06, pad=0.06, shrink=0.7).set_label('OFA obs (% saturation)')
        fig.suptitle(f'{region_name} | {date} cycle {cycle:02d} | {platform}\n{stats_str}', fontsize=10)


for name, (lat0, lon0, lat1, lon1) in ALL_REGIONS.items():
    plot_regional_windows(name, lat0, lon0, lat1, lon1)


## 13. Why does Legacy touch fewer tiles? Raw footprint sampling

Section 12's first window (Oklahoma, 2020-06-01, cycle 01, Metop-A) showed Legacy hitting 21
tiles vs H121's 64 -- on the *same* M36 grid. Hypothesis: this isn't a coverage difference (same
instrument, same orbit), it's a sampling/discretization mismatch. Legacy BUFR's native footprint
spacing (~25 km) is close to the 36 km M36 tile size, so a satellite track sweeping diagonally
across the tile grid leaves some tiles with zero footprint centers purely by chance, even though
the swath physically covers them. H121's denser 12.5 km DGG doesn't have this problem.

Check this directly by reading the *raw* (pre-superob) point locations for this exact
date/cycle/region and plotting them against the M36 tile boundaries. Raw files are only
available locally for 2020-06-01..10, which is why this window was usable.

**Note:** uses default QC (`QC_DEFAULT_BUFR`/`QC_DEFAULT_H121` from `lib.qc`), not the revised
H121 QC -- the goal here is to see *where* native-resolution footprints land, not to reproduce
the exact OFA-ingested set.

In [ ]:
from datetime import datetime
from lib.readers import read_bufr, read_h121
from lib.qc import QC_DEFAULT_BUFR, QC_DEFAULT_H121

DIAG_DATE = datetime(2020, 6, 1)
DIAG_WINDOW = 1  # matches 'cycle 01' in the section 12 example
DIAG_LAT0, DIAG_LON0, DIAG_LAT1, DIAG_LON1 = 35, -99, 38, -96
DIAG_PAD = 0.5

OBS_ROOT = Path('/Users/amfox/Desktop/ASCAT_SSM_CDR/discover_sample')
DIAG_DOMAIN = (DIAG_LAT0 - DIAG_PAD, DIAG_LON0 - DIAG_PAD, DIAG_LAT1 + DIAG_PAD, DIAG_LON1 + DIAG_PAD)

raw_legacy = read_bufr(
    str(OBS_ROOT / 'legacy_bufr' / 'metop_a' / f'Y{DIAG_DATE:%Y}' / f'M{DIAG_DATE:%m}'),
    DIAG_DATE, 'M02-ASCA-ASCSMO02-NA-5.0-', domain=DIAG_DOMAIN, qc=QC_DEFAULT_BUFR,
)
raw_h121 = read_h121(
    str(OBS_ROOT / 'H121' / 'metop_a' / f'Y{DIAG_DATE:%Y}' / f'M{DIAG_DATE:%m}'),
    DIAG_DATE, domain=DIAG_DOMAIN, qc=QC_DEFAULT_H121,
)

legacy_win = {k: v[raw_legacy['window'] == DIAG_WINDOW] for k, v in raw_legacy.items()}
h121_win = {k: v[raw_h121['window'] == DIAG_WINDOW] for k, v in raw_h121.items()}
print(f"Legacy raw points in window: {len(legacy_win['lat']):,}")
print(f"H121 raw points in window:   {len(h121_win['lat']):,}")


In [ ]:
diag_tiles = tile_grid_df[
    tile_grid_df['tile_lat'].between(DIAG_LAT0, DIAG_LAT1) & tile_grid_df['tile_lon'].between(DIAG_LON0, DIAG_LON1)
]

fig, axes = plt.subplots(1, 2, figsize=(11, 5), subplot_kw={'projection': ccrs.PlateCarree()}, constrained_layout=True)
for ax, (label, pts) in zip(axes, [('Legacy raw footprints', legacy_win), ('H121 raw footprints', h121_win)]):
    _setup_region_ax(ax, DIAG_LAT0, DIAG_LON0, DIAG_LAT1, DIAG_LON1, f'{label} (n={len(pts["lat"]):,})')
    patches = [mpatches.Polygon(np.column_stack(ij_to_corners(int(r.tile_i), int(r.tile_j))), closed=True)
               for r in diag_tiles.itertuples()]
    ax.add_collection(PatchCollection(patches, facecolor='none', edgecolor='0.6', linewidth=0.5,
                                       transform=ccrs.PlateCarree(), zorder=2))
    ax.scatter(pts['lon'], pts['lat'], s=10, color='#d62728', edgecolor='black', linewidth=0.3,
               transform=ccrs.PlateCarree(), zorder=5)
fig.suptitle(f'{DIAG_DATE:%Y-%m-%d} window {DIAG_WINDOW} | raw footprint locations vs M36 tile grid')


**Reading the plot:** if the hypothesis is right, Legacy's red dots should be visibly sparser
relative to the tile grid lines -- with whole tiles the swath crosses left empty -- while H121's
dots should densely fill nearly every tile the swath touches.

**Window-boundary check:** if the missing northern footprints in Legacy's window 1 are a
cycle-assignment artifact (asymmetric `[low, up)` vs `(low, up]` interval convention between
the two readers), they should show up in an *adjacent* window (0 or 2) instead of being lost
entirely. Plot windows 0, 1, 2 for both products to check.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8.5), subplot_kw={'projection': ccrs.PlateCarree()}, constrained_layout=True)
for row, (label, raw) in enumerate([('Legacy', raw_legacy), ('H121', raw_h121)]):
    for col, win in enumerate([0, 1, 2]):
        pts = {k: v[raw['window'] == win] for k, v in raw.items()}
        ax = axes[row, col]
        _setup_region_ax(ax, DIAG_LAT0, DIAG_LON0, DIAG_LAT1, DIAG_LON1, f'{label} window {win} (n={len(pts["lat"]):,})' if row == 0 else f'window {win} (n={len(pts["lat"]):,})')
        patches = [mpatches.Polygon(np.column_stack(ij_to_corners(int(r.tile_i), int(r.tile_j))), closed=True)
                   for r in diag_tiles.itertuples()]
        ax.add_collection(PatchCollection(patches, facecolor='none', edgecolor='0.6', linewidth=0.5,
                                           transform=ccrs.PlateCarree(), zorder=2))
        ax.scatter(pts['lon'], pts['lat'], s=8, color='#d62728', edgecolor='black', linewidth=0.3,
                   transform=ccrs.PlateCarree(), zorder=5)
fig.suptitle(f'{DIAG_DATE:%Y-%m-%d} | windows 0-2 | raw footprint locations vs M36 tile grid')


**Reading this:** if the northern gap in Legacy window 1 reappears (fully or partly) in window
0 or window 2, that confirms the cycle-assignment boundary is splitting one continuous overpass
across windows differently between the two readers. If the northern area is also empty in
windows 0 and 2, the swath simply didn't cover it for Legacy at all in this 3-window span, and
the gap is a genuine sampling/coverage difference instead.

**Sensitivity check:** the swath-edge explanation is a guess, not a measurement. A more direct
test: re-read H121 for this same window at `sens_min` = 1.0 (current default), 2, 3, 4, 5 and
see whether the northern area H121 covers but Legacy doesn't is specifically low-sensitivity,
marginal retrievals -- i.e. whether tightening sensitivity QC shrinks H121's footprint toward
Legacy's pattern as the threshold rises.

In [ ]:
SENS_THRESHOLDS = [1.0, 2.0, 3.0, 4.0, 5.0]

h121_by_sens = {}
for thresh in SENS_THRESHOLDS:
    raw = read_h121(
        str(OBS_ROOT / 'H121' / 'metop_a' / f'Y{DIAG_DATE:%Y}' / f'M{DIAG_DATE:%m}'),
        DIAG_DATE, domain=DIAG_DOMAIN, qc={**QC_DEFAULT_H121, 'sens_min': thresh},
    )
    h121_by_sens[thresh] = {k: v[raw['window'] == DIAG_WINDOW] for k, v in raw.items()}
    print(f"H121 raw points, sens_min={thresh}: {len(h121_by_sens[thresh]['lat']):,}")

fig, axes = plt.subplots(2, 3, figsize=(15, 8.5), subplot_kw={'projection': ccrs.PlateCarree()}, constrained_layout=True)
patches = [mpatches.Polygon(np.column_stack(ij_to_corners(int(r.tile_i), int(r.tile_j))), closed=True)
           for r in diag_tiles.itertuples()]

panels = [('Legacy (n={})'.format(len(legacy_win['lat'])), legacy_win)] + [
    (f'H121 sens_min={t} (n={len(h121_by_sens[t]["lat"]):,})', h121_by_sens[t]) for t in SENS_THRESHOLDS
]
for ax, (title, pts) in zip(axes.flat, panels):
    _setup_region_ax(ax, DIAG_LAT0, DIAG_LON0, DIAG_LAT1, DIAG_LON1, title)
    ax.add_collection(PatchCollection(patches, facecolor='none', edgecolor='0.6', linewidth=0.5,
                                       transform=ccrs.PlateCarree(), zorder=2))
    ax.scatter(pts['lon'], pts['lat'], s=10, color='#d62728', edgecolor='black', linewidth=0.3,
               transform=ccrs.PlateCarree(), zorder=5)
fig.suptitle(f'{DIAG_DATE:%Y-%m-%d} window {DIAG_WINDOW} | H121 raw obs locations across sensitivity thresholds, vs Legacy')


**Sensitivity is the wrong direction -- check Legacy's own QC instead:** points actually drop
out from the south/southeast as `sens_min` rises, and the last survivors at `sens_min=5` sit in
the north -- the opposite of what "marginal sensitivity explains the gap" would predict. So the
gap isn't an H121 sensitivity effect. Test directly whether Legacy's QC (topographic complexity,
wetland fraction, processing/correction flags) is rejecting real northern footprints, or whether
those footprints are simply absent from the raw BUFR file regardless of QC.

In [ ]:
legacy_noqc = read_bufr(
    str(OBS_ROOT / 'legacy_bufr' / 'metop_a' / f'Y{DIAG_DATE:%Y}' / f'M{DIAG_DATE:%m}'),
    DIAG_DATE, 'M02-ASCA-ASCSMO02-NA-5.0-', domain=DIAG_DOMAIN,
    qc={'ssm_min': None, 'ssm_max': None, 'smpf_ok': None, 'smcf_ok': None, 'tpcx_max': None, 'iwfr_max': None},
)
legacy_noqc_win = {k: v[legacy_noqc['window'] == DIAG_WINDOW] for k, v in legacy_noqc.items()}
print(f"Legacy raw points, QC ON (default):  {len(legacy_win['lat']):,}")
print(f"Legacy raw points, QC OFF entirely:  {len(legacy_noqc_win['lat']):,}")

fig, axes = plt.subplots(1, 2, figsize=(10, 5), subplot_kw={'projection': ccrs.PlateCarree()}, constrained_layout=True)
for ax, (title, pts) in zip(axes, [
    (f'Legacy, QC on (n={len(legacy_win["lat"])})', legacy_win),
    (f'Legacy, QC off (n={len(legacy_noqc_win["lat"])})', legacy_noqc_win),
]):
    _setup_region_ax(ax, DIAG_LAT0, DIAG_LON0, DIAG_LAT1, DIAG_LON1, title)
    ax.add_collection(PatchCollection(patches, facecolor='none', edgecolor='0.6', linewidth=0.5,
                                       transform=ccrs.PlateCarree(), zorder=2))
    ax.scatter(pts['lon'], pts['lat'], s=10, color='#d62728', edgecolor='black', linewidth=0.3,
               transform=ccrs.PlateCarree(), zorder=5)
fig.suptitle(f'{DIAG_DATE:%Y-%m-%d} window {DIAG_WINDOW} | does disabling Legacy QC reveal northern footprints?')


**Reading this:** if QC-off recovers northern footprints that QC-on was rejecting, the gap is
a real Legacy QC effect (terrain/wetland/flag-driven) -- worth tracking down which specific
filter is responsible. If QC-off still shows nothing in the north, the raw BUFR file genuinely
has no footprints there for this overpass, and the explanation has to be upstream of QC entirely
(swath/file coverage).

## 13b. Which Legacy QC filter is rejecting the north?

Four candidate filters in `QC_DEFAULT_BUFR`: `smpf_ok` (processing flag), `smcf_ok` (correction
flag), `tpcx_max` (topographic complexity), `iwfr_max` (inundation/wetland fraction). Re-read
with only one filter enabled at a time (the other three off) to see which one alone reproduces
the northern gap.

In [ ]:
QC_ALL_OFF = {'ssm_min': None, 'ssm_max': None, 'smpf_ok': None, 'smcf_ok': None, 'tpcx_max': None, 'iwfr_max': None}

filter_variants = {
    'smpf_ok (processing flag)': {**QC_ALL_OFF, 'smpf_ok': QC_DEFAULT_BUFR['smpf_ok']},
    'smcf_ok (correction flag)': {**QC_ALL_OFF, 'smcf_ok': QC_DEFAULT_BUFR['smcf_ok']},
    'tpcx_max (topo complexity)': {**QC_ALL_OFF, 'tpcx_max': QC_DEFAULT_BUFR['tpcx_max']},
    'iwfr_max (wetland fraction)': {**QC_ALL_OFF, 'iwfr_max': QC_DEFAULT_BUFR['iwfr_max']},
}

legacy_by_filter = {}
for label, qc in filter_variants.items():
    raw = read_bufr(
        str(OBS_ROOT / 'legacy_bufr' / 'metop_a' / f'Y{DIAG_DATE:%Y}' / f'M{DIAG_DATE:%m}'),
        DIAG_DATE, 'M02-ASCA-ASCSMO02-NA-5.0-', domain=DIAG_DOMAIN, qc=qc,
    )
    legacy_by_filter[label] = {k: v[raw['window'] == DIAG_WINDOW] for k, v in raw.items()}
    print(f"{label}: n={len(legacy_by_filter[label]['lat']):,}")

fig, axes = plt.subplots(1, 5, figsize=(22, 5), subplot_kw={'projection': ccrs.PlateCarree()}, constrained_layout=True)
panels = [('QC off, all (n={})'.format(len(legacy_noqc_win['lat'])), legacy_noqc_win)] + [
    (f'only {label} (n={len(pts["lat"])})', pts) for label, pts in legacy_by_filter.items()
]
for ax, (title, pts) in zip(axes, panels):
    _setup_region_ax(ax, DIAG_LAT0, DIAG_LON0, DIAG_LAT1, DIAG_LON1, title)
    ax.add_collection(PatchCollection(patches, facecolor='none', edgecolor='0.6', linewidth=0.5,
                                       transform=ccrs.PlateCarree(), zorder=2))
    ax.scatter(pts['lon'], pts['lat'], s=10, color='#d62728', edgecolor='black', linewidth=0.3,
               transform=ccrs.PlateCarree(), zorder=5)
fig.suptitle(f'{DIAG_DATE:%Y-%m-%d} window {DIAG_WINDOW} | isolating which Legacy QC filter removes the north')


**Reading this:** whichever single-filter panel reproduces the QC-on gap (n=67, footprints
confined to the south) is the filter responsible. If none of them alone reproduces it, the
combination is responsible (or two filters are doing similar but not identical work).

## 14. Which product is more biased relative to the model background?

Section 9's bias map (and `legacy_vs_h121_highlat_bias.ipynb`) showed Legacy and H121 disagree
sharply above ~50N -- but that only tells us the two products disagree, not which one is
wrong. Treating the Catchment model background (`fcst`) as the closer-to-truth reference,
compare each product's own innovation (O-F) by latitude band, *separately* (not their
difference). If one product's innovation blows up at high latitude relative to its own
mid-latitude baseline while the other doesn't, that product is the one diverging from the
model -- regardless of which one reads "wetter" or "drier" than the other.

**Caveat:** this is unscaled monitor-mode O-F, so the absolute level everywhere includes a
baseline unit-mismatch offset (% saturation obs vs volumetric-fraction model background) --
not meaningful on its own. What's meaningful is the *change* from each product's mid-latitude
baseline to its high-latitude value.

In [ ]:
INNOV_LAT_BINS = np.arange(-60, 91, 10)
ofa['lat_bin'] = pd.cut(ofa['tile_lat'], bins=INNOV_LAT_BINS)

innov_by_product = {}
for product in ['legacy', 'h121']:
    sub = ofa[ofa['product'] == product]
    g = sub.groupby('lat_bin', observed=True)['innov_pct'].agg(['mean', 'median', 'std', 'size'])
    bounds = pd.Series(g.index.astype(str)).str.strip('()[]').str.split(',', expand=True).astype(float)
    g = g.reset_index()
    g['lat_mid'] = bounds.mean(axis=1).to_numpy()
    innov_by_product[product] = g.sort_values('lat_mid')

for product, g in innov_by_product.items():
    print(f'=== {product} ===')
    display(g[['lat_bin', 'mean', 'median', 'size']])


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for product, color in [('legacy', '#1f77b4'), ('h121', '#d62728')]:
    g = innov_by_product[product]
    ax.plot(g['lat_mid'], g['mean'], 'o-', label=product, color=color)
ax.set_xlabel('Tile latitude (deg N)')
ax.set_ylabel('Mean innovation O-F (% sat, unscaled monitor-mode)')
ax.set_title('Legacy vs H121 innovation by latitude (each vs the same Catchment background)')
ax.legend()
ax.grid(True, linewidth=0.25, alpha=0.4)
fig.tight_layout()


**Reading this:** Legacy's innovation roughly doubles from its 30-50N baseline (~19) to
50-80N (~40-50) -- a jump of +25 to +30. H121's innovation barely moves: it dips at 50-60N
before rising modestly to ~23-25 at 60-80N, an increase of roughly +10 over its own baseline.

**Legacy is the one that diverges from the Catchment background at high latitude, not H121.**
If the model is closer to truth, H121's high-latitude values are actually more consistent with
it than Legacy's are -- the opposite of what the raw Legacy-vs-H121 disagreement alone would
suggest. This doesn't prove H121 is "right" (Catchment's own snow/frozen-soil physics could
share a similar high-latitude blind spot), but it means the bias found in
`legacy_vs_h121_highlat_bias.ipynb` should not be assumed to be "H121's problem" without this
check -- by this metric it looks more like Legacy's.

## 15. Summary

In [ ]:
summary_rows = []
for plat in platforms:
    legacy_n = int(pivot.loc[plat, 'legacy'])
    h121_n = int(pivot.loc[plat, 'h121'])
    summary_rows.append({
        'platform': plat,
        'legacy_obs': legacy_n,
        'h121_obs': h121_n,
        'h121_to_legacy_ratio': round(h121_n / legacy_n, 2),
    })
summary_df = pd.DataFrame(summary_rows)
display(summary_df)
print(f"Mean coverage fraction: legacy={cov_legacy['coverage_frac'].mean():.2f}, "
      f"h121={cov_h121['coverage_frac'].mean():.2f}")
